# Hourly energy-system simulation

This first model uses fixed plant sizes and rule-based dispatch. It balances the hourly scaled electricity demand, including the truck electricity series, and the separate truck methanol demand. It is not yet a full balance of every sector in the project's annual 730 PJ total.

Run `plotting.ipynb` through its trucking-demand cell first. This notebook reads `data/ProductionConsumptionSettlement_DK_modified.csv`. The local `HourDK` timestamp repeats or skips an hour at daylight-saving transitions, so rows are sorted by the unique `HourUTC` field and checked for one-hour intervals.

The current input series represent 480 PJ of scaled direct electricity demand, about 7.12 PJ of truck electricity demand, about 35.42 PJ of truck methanol demand, and 580 PJ of scaled wind-plus-solar electricity. These are current notebook assumptions, not proof that the full energy system is balanced. Other fuel demands, including light transport, shipping, and aviation, are outside this first simulation boundary.

Renewables serve direct electricity demand first. Deficits are served by battery discharge, then biomass generation, then counted as unserved electricity. Renewable surplus is allocated between battery charging and the P2X plant according to the configurable priority. P2X uses renewable surplus only in this first version. Methanol production is limited by P2X input power and tank headroom; hourly methanol demand is served from same-hour production and stored methanol.

Scenario capacities and efficiencies below are illustrative placeholders to exercise the model. Replace them with project assumptions before interpreting results. Biomass availability is tracked as biomass fuel input energy; generator output is limited by its power capacity and conversion efficiency.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import total_annual_energy_consumption_dk as taec

DATA_PATH = Path.cwd() / 'data' / 'ProductionConsumptionSettlement_DK_modified.csv'
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f'Input not found: {DATA_PATH}. Run plotting.ipynb through the trucking-demand cell first.'
    )

data = pd.read_csv(DATA_PATH, sep=';', decimal=',')
required_columns = [
    'HourUTC',
    'HourDK',
    'TotalRenewableScaledMWh',
    'GrossConsumptionScaledMWh',
    'TrucksElectricityDemandMWh',
    'TrucksMethanolDemandMWh',
]
missing_columns = [column for column in required_columns if column not in data.columns]
if missing_columns:
    raise ValueError(f'Missing simulation input columns: {missing_columns}')
if data.empty:
    raise ValueError('The simulation input contains no hourly rows')

data['HourUTC'] = pd.to_datetime(data['HourUTC'], errors='raise')
data['HourDK'] = pd.to_datetime(data['HourDK'], errors='raise')
if data['HourUTC'].isna().any() or data['HourDK'].isna().any():
    raise ValueError('Simulation timestamps cannot be missing')
if not data['HourUTC'].is_unique:
    raise ValueError('HourUTC must uniquely identify each settlement interval')
data = data.sort_values('HourUTC', kind='stable').reset_index(drop=True)
if not data['HourUTC'].diff().dropna().eq(pd.Timedelta(hours=1)).all():
    raise ValueError('Expected consecutive one-hour settlement intervals in HourUTC')

input_energy_columns = required_columns[2:]
data[input_energy_columns] = data[input_energy_columns].apply(pd.to_numeric, errors='raise')
if not np.isfinite(data[input_energy_columns].to_numpy(dtype=float)).all():
    raise ValueError('Simulation energy inputs must be finite')
if (data[input_energy_columns] < 0).any().any():
    raise ValueError('Simulation energy inputs cannot be negative')

data['ElectricityDemandMWh'] = (
    data['GrossConsumptionScaledMWh'] + data['TrucksElectricityDemandMWh']
)
data['MethanolDemandMWh'] = data['TrucksMethanolDemandMWh']

PJ_TO_MWH = 1_000_000 / 3.6
expected_truck_electricity_mwh = taec.TRUCKS_ELEC_DEMAND * PJ_TO_MWH
expected_truck_methanol_mwh = taec.TRUCKS_METH_DEMAND * PJ_TO_MWH
if not np.isclose(data['TrucksElectricityDemandMWh'].sum(), expected_truck_electricity_mwh):
    raise ValueError('Hourly truck electricity demand does not match total_annual_energy_consumption_dk.py')
if not np.isclose(data['TrucksMethanolDemandMWh'].sum(), expected_truck_methanol_mwh):
    raise ValueError('Hourly truck methanol demand does not match total_annual_energy_consumption_dk.py')

print(f'Loaded {len(data):,} hourly intervals from {data["HourDK"].min()} to {data["HourDK"].max()}')
print(f'Truck electricity demand: {data["TrucksElectricityDemandMWh"].sum() / PJ_TO_MWH:.2f} PJ')
print(f'Truck methanol demand: {data["TrucksMethanolDemandMWh"].sum() / PJ_TO_MWH:.2f} PJ')

## Scenario inputs

P2X capacity is electrical input power in MW. Battery capacity is specified separately as power in MW and stored energy in MWh. Methanol tank capacity is stored methanol energy in MWh (fuel output basis), not an electricity capacity. The explicit `METHANOL_STORAGE_CAPACITY_MWH` input is initialized to ten days of average demand as an example. Battery and tank start empty for this run; ending inventories are reported so that stock changes are visible.

The example plant sizes, efficiencies, and ten-day methanol tank are placeholders only. `surplus_priority` can be set to `battery_first` or `p2x_first` to compare how surplus renewable electricity is used.

In [ ]:
METHANOL_STORAGE_DAYS = 10
average_methanol_demand_mwh_per_hour = data['MethanolDemandMWh'].mean()
METHANOL_STORAGE_CAPACITY_MWH = (
    average_methanol_demand_mwh_per_hour * 24 * METHANOL_STORAGE_DAYS
)

SCENARIO = {
    'p2x_input_capacity_mw': 3_000.0,
    'p2x_efficiency': 0.55,
    'battery_power_capacity_mw': 5_000.0,
    'battery_energy_capacity_mwh': 100_000.0,
    'battery_charge_efficiency': 0.95,
    'battery_discharge_efficiency': 0.95,
    'battery_initial_soc_mwh': 0.0,
    'methanol_tank_capacity_mwh': METHANOL_STORAGE_CAPACITY_MWH,
    'methanol_initial_inventory_mwh': 0.0,
    'biomass_power_capacity_mw': 5_000.0,
    'biomass_electric_efficiency': 0.35,
    'biomass_available_pj': float(taec.AVAILABLE_BIOMASS),
    'surplus_priority': 'p2x_first',
}

print(f'Methanol tank capacity: {SCENARIO["methanol_tank_capacity_mwh"]:,.0f} MWh')

## Dispatch and hourly state

Each row is one hour, so a power limit in MW is also the maximum MWh input/output for that interval. Battery charging consumes surplus electricity and adds charge-efficiency-adjusted energy to its state of charge. Battery discharge removes more stored energy than it delivers to the grid. Biomass is used only for direct electricity deficits; this version does not use biomass electricity to run P2X.

In [ ]:
def run_simulation(input_data: pd.DataFrame, config: dict[str, float | str]) -> pd.DataFrame:
    required_config = {
        'p2x_input_capacity_mw',
        'p2x_efficiency',
        'battery_power_capacity_mw',
        'battery_energy_capacity_mwh',
        'battery_charge_efficiency',
        'battery_discharge_efficiency',
        'battery_initial_soc_mwh',
        'methanol_tank_capacity_mwh',
        'methanol_initial_inventory_mwh',
        'biomass_power_capacity_mw',
        'biomass_electric_efficiency',
        'biomass_available_pj',
        'surplus_priority',
    }
    missing_config = required_config.difference(config)
    if missing_config:
        raise ValueError(f'Missing scenario settings: {sorted(missing_config)}')

    numeric_config = required_config - {'surplus_priority'}
    for name in numeric_config:
        value = config[name]
        if not isinstance(value, (int, float)) or not np.isfinite(value) or value < 0:
            raise ValueError(f'{name} must be a finite, non-negative number')
    for name in ('p2x_efficiency', 'battery_charge_efficiency', 'battery_discharge_efficiency', 'biomass_electric_efficiency'):
        if not 0 < config[name] <= 1:
            raise ValueError(f'{name} must be greater than zero and at most one')
    if config['surplus_priority'] not in {'battery_first', 'p2x_first'}:
        raise ValueError("surplus_priority must be 'battery_first' or 'p2x_first'")

    battery_capacity = float(config['battery_energy_capacity_mwh'])
    battery_soc = float(config['battery_initial_soc_mwh'])
    tank_capacity = float(config['methanol_tank_capacity_mwh'])
    methanol_inventory = float(config['methanol_initial_inventory_mwh'])
    if battery_soc > battery_capacity:
        raise ValueError('Initial battery state of charge exceeds battery energy capacity')
    if methanol_inventory > tank_capacity:
        raise ValueError('Initial methanol inventory exceeds tank capacity')
    if not input_data['HourUTC'].is_monotonic_increasing:
        raise ValueError('Input rows must be sorted by HourUTC before simulation')

    biomass_remaining_mwh_fuel = float(config['biomass_available_pj']) * PJ_TO_MWH
    records = []
    time_step_hours = 1.0

    def p2x_dispatch(available_electricity_mwh: float, methanol_demand_mwh: float) -> tuple[float, float]:
        possible_methanol_output = max(0.0, tank_capacity - methanol_inventory + methanol_demand_mwh)
        input_for_tank_room = possible_methanol_output / float(config['p2x_efficiency'])
        p2x_input = min(
            available_electricity_mwh,
            float(config['p2x_input_capacity_mw']) * time_step_hours,
            input_for_tank_room,
        )
        return p2x_input, p2x_input * float(config['p2x_efficiency'])

    selected_columns = [
        'HourUTC',
        'HourDK',
        'TotalRenewableScaledMWh',
        'ElectricityDemandMWh',
        'MethanolDemandMWh',
    ]
    for hour_utc, hour_dk, renewable_mwh, electricity_demand_mwh, methanol_demand_mwh in input_data[selected_columns].itertuples(index=False, name=None):
        battery_soc_start = battery_soc
        methanol_inventory_start = methanol_inventory
        renewable_to_load = min(renewable_mwh, electricity_demand_mwh)
        battery_discharge_mwh = 0.0
        biomass_generation_mwh = 0.0
        biomass_fuel_input_mwh = 0.0
        battery_charge_input_mwh = 0.0
        p2x_electricity_input_mwh = 0.0
        methanol_production_mwh = 0.0
        curtailed_renewable_mwh = 0.0
        unserved_electricity_mwh = 0.0

        if renewable_mwh < electricity_demand_mwh:
            deficit_mwh = electricity_demand_mwh - renewable_mwh
            battery_discharge_mwh = min(
                deficit_mwh,
                float(config['battery_power_capacity_mw']) * time_step_hours,
                battery_soc * float(config['battery_discharge_efficiency']),
            )
            battery_soc -= battery_discharge_mwh / float(config['battery_discharge_efficiency'])
            deficit_mwh -= battery_discharge_mwh

            biomass_generation_mwh = min(
                deficit_mwh,
                float(config['biomass_power_capacity_mw']) * time_step_hours,
                biomass_remaining_mwh_fuel * float(config['biomass_electric_efficiency']),
            )
            biomass_fuel_input_mwh = biomass_generation_mwh / float(config['biomass_electric_efficiency'])
            biomass_remaining_mwh_fuel -= biomass_fuel_input_mwh
            deficit_mwh -= biomass_generation_mwh
            unserved_electricity_mwh = deficit_mwh
            electricity_served_mwh = electricity_demand_mwh - unserved_electricity_mwh
        else:
            electricity_served_mwh = electricity_demand_mwh
            surplus_mwh = renewable_mwh - electricity_demand_mwh
            if config['surplus_priority'] == 'p2x_first':
                p2x_electricity_input_mwh, methanol_production_mwh = p2x_dispatch(
                    surplus_mwh, methanol_demand_mwh
                )
                surplus_mwh -= p2x_electricity_input_mwh

            battery_charge_input_mwh = min(
                surplus_mwh,
                float(config['battery_power_capacity_mw']) * time_step_hours,
                max(0.0, (battery_capacity - battery_soc) / float(config['battery_charge_efficiency'])),
            )
            battery_soc += battery_charge_input_mwh * float(config['battery_charge_efficiency'])
            surplus_mwh -= battery_charge_input_mwh

            if config['surplus_priority'] == 'battery_first':
                p2x_electricity_input_mwh, methanol_production_mwh = p2x_dispatch(
                    surplus_mwh, methanol_demand_mwh
                )
                surplus_mwh -= p2x_electricity_input_mwh
            curtailed_renewable_mwh = surplus_mwh

        methanol_available_mwh = methanol_inventory + methanol_production_mwh
        methanol_served_mwh = min(methanol_demand_mwh, methanol_available_mwh)
        unserved_methanol_mwh = methanol_demand_mwh - methanol_served_mwh
        methanol_inventory = methanol_available_mwh - methanol_served_mwh

        tolerance = 1e-6
        if battery_soc < -tolerance or battery_soc > battery_capacity + tolerance:
            raise RuntimeError('Battery state exceeded its configured energy bounds')
        if methanol_inventory < -tolerance or methanol_inventory > tank_capacity + tolerance:
            raise RuntimeError('Methanol inventory exceeded tank capacity bounds')
        battery_soc = min(battery_capacity, max(0.0, battery_soc))
        methanol_inventory = min(tank_capacity, max(0.0, methanol_inventory))

        records.append({
            'HourUTC': hour_utc,
            'HourDK': hour_dk,
            'RenewableGenerationMWh': renewable_mwh,
            'RenewableToLoadMWh': renewable_to_load,
            'ElectricityDemandMWh': electricity_demand_mwh,
            'ElectricityServedMWh': electricity_served_mwh,
            'UnservedElectricityMWh': unserved_electricity_mwh,
            'BatteryChargeInputMWh': battery_charge_input_mwh,
            'BatteryDischargeOutputMWh': battery_discharge_mwh,
            'BatterySOCStartMWh': battery_soc_start,
            'BatterySOCEndMWh': battery_soc,
            'BiomassGenerationMWh': biomass_generation_mwh,
            'BiomassFuelInputMWh': biomass_fuel_input_mwh,
            'P2XElectricityInputMWh': p2x_electricity_input_mwh,
            'RenewableCurtailmentMWh': curtailed_renewable_mwh,
            'MethanolDemandMWh': methanol_demand_mwh,
            'MethanolProductionMWh': methanol_production_mwh,
            'MethanolServedMWh': methanol_served_mwh,
            'UnservedMethanolMWh': unserved_methanol_mwh,
            'MethanolInventoryStartMWh': methanol_inventory_start,
            'MethanolInventoryEndMWh': methanol_inventory,
        })

    results = pd.DataFrame.from_records(records)
    electrical_supply = (
        results['RenewableGenerationMWh']
        + results['BatteryDischargeOutputMWh']
        + results['BiomassGenerationMWh']
    )
    electrical_use = (
        results['ElectricityServedMWh']
        + results['BatteryChargeInputMWh']
        + results['P2XElectricityInputMWh']
        + results['RenewableCurtailmentMWh']
    )
    if not np.allclose(electrical_supply, electrical_use, rtol=1e-9, atol=1e-6):
        raise RuntimeError('Hourly electricity balance did not close')
    methanol_balance = (
        results['MethanolInventoryStartMWh']
        + results['MethanolProductionMWh']
        - results['MethanolServedMWh']
        - results['MethanolInventoryEndMWh']
    )
    if not np.allclose(methanol_balance, 0.0, rtol=1e-9, atol=1e-6):
        raise RuntimeError('Hourly methanol inventory balance did not close')
    biomass_budget_mwh = float(config['biomass_available_pj']) * PJ_TO_MWH
    if results['BiomassFuelInputMWh'].sum() > biomass_budget_mwh + 1e-6:
        raise RuntimeError('Biomass use exceeded the configured annual fuel budget')
    return results


simulation_results = run_simulation(data, SCENARIO)
print(f'Simulated {len(simulation_results):,} hourly intervals')

## Dispatch sanity checks

These small cases check same-hour P2X supply with no tank, tank capacity limits, finite biomass fuel, and battery discharge efficiency.

In [ ]:
def one_hour_case(renewable_mwh: float, electricity_demand_mwh: float, methanol_demand_mwh: float) -> pd.DataFrame:
    return pd.DataFrame({
        'HourUTC': [pd.Timestamp('2025-01-01 00:00:00')],
        'HourDK': [pd.Timestamp('2025-01-01 01:00:00')],
        'TotalRenewableScaledMWh': [renewable_mwh],
        'ElectricityDemandMWh': [electricity_demand_mwh],
        'MethanolDemandMWh': [methanol_demand_mwh],
    })


def test_config(**overrides: float | str) -> dict[str, float | str]:
    config = dict(SCENARIO)
    config.update({
        'p2x_input_capacity_mw': 0.0,
        'p2x_efficiency': 0.5,
        'battery_power_capacity_mw': 0.0,
        'battery_energy_capacity_mwh': 0.0,
        'battery_initial_soc_mwh': 0.0,
        'methanol_tank_capacity_mwh': 0.0,
        'methanol_initial_inventory_mwh': 0.0,
        'biomass_power_capacity_mw': 0.0,
        'biomass_electric_efficiency': 0.5,
        'biomass_available_pj': 0.0,
        'surplus_priority': 'p2x_first',
    })
    config.update(overrides)
    return config


direct_methanol = run_simulation(
    one_hour_case(renewable_mwh=2.0, electricity_demand_mwh=1.0, methanol_demand_mwh=0.5),
    test_config(p2x_input_capacity_mw=2.0),
)
assert np.isclose(direct_methanol['MethanolProductionMWh'].iloc[0], 0.5)
assert np.isclose(direct_methanol['UnservedMethanolMWh'].iloc[0], 0.0)
assert np.isclose(direct_methanol['MethanolInventoryEndMWh'].iloc[0], 0.0)

tank_limited = run_simulation(
    one_hour_case(renewable_mwh=10.0, electricity_demand_mwh=0.0, methanol_demand_mwh=0.0),
    test_config(p2x_input_capacity_mw=5.0, methanol_tank_capacity_mwh=2.0),
)
assert np.isclose(tank_limited['MethanolInventoryEndMWh'].iloc[0], 2.0)
assert np.isclose(tank_limited['RenewableCurtailmentMWh'].iloc[0], 6.0)

biomass_limited = run_simulation(
    one_hour_case(renewable_mwh=0.0, electricity_demand_mwh=1.0, methanol_demand_mwh=0.0),
    test_config(biomass_power_capacity_mw=10.0, biomass_available_pj=3.6e-6),
)
assert np.isclose(biomass_limited['BiomassFuelInputMWh'].sum(), 1.0)
assert np.isclose(biomass_limited['BiomassGenerationMWh'].sum(), 0.5)
assert np.isclose(biomass_limited['UnservedElectricityMWh'].sum(), 0.5)

battery_limited = run_simulation(
    one_hour_case(renewable_mwh=0.0, electricity_demand_mwh=2.0, methanol_demand_mwh=0.0),
    test_config(
        battery_power_capacity_mw=2.0,
        battery_energy_capacity_mwh=2.0,
        battery_initial_soc_mwh=2.0,
        battery_discharge_efficiency=0.9,
    ),
)
assert np.isclose(battery_limited['BatteryDischargeOutputMWh'].iloc[0], 1.8)
assert np.isclose(battery_limited['UnservedElectricityMWh'].iloc[0], 0.2)
assert np.isclose(battery_limited['BatterySOCEndMWh'].iloc[0], 0.0)

battery_first = run_simulation(
    one_hour_case(renewable_mwh=10.0, electricity_demand_mwh=0.0, methanol_demand_mwh=0.0),
    test_config(
        p2x_input_capacity_mw=10.0,
        battery_power_capacity_mw=4.0,
        battery_energy_capacity_mwh=4.0,
        battery_charge_efficiency=1.0,
        methanol_tank_capacity_mwh=2.0,
        surplus_priority='battery_first',
    ),
)
assert np.isclose(battery_first['BatteryChargeInputMWh'].iloc[0], 4.0)
assert np.isclose(battery_first['P2XElectricityInputMWh'].iloc[0], 4.0)
assert np.isclose(battery_first['RenewableCurtailmentMWh'].iloc[0], 2.0)

print('All dispatch sanity checks passed')

In [ ]:
def mwh_to_pj(energy_mwh: float) -> float:
    return energy_mwh * 3.6e-6


electricity_demand_mwh = simulation_results['ElectricityDemandMWh'].sum()
methanol_demand_mwh = simulation_results['MethanolDemandMWh'].sum()
biomass_fuel_used_mwh = simulation_results['BiomassFuelInputMWh'].sum()
renewable_production_mwh = data['TotalRenewableScaledMWh'].sum()
p2x_input_mwh = simulation_results['P2XElectricityInputMWh'].sum()
methanol_production_mwh = simulation_results['MethanolProductionMWh'].sum()

summary = pd.Series({
    'Electricity demand (PJ)': mwh_to_pj(electricity_demand_mwh),
    'Unserved electricity (MWh)': simulation_results['UnservedElectricityMWh'].sum(),
    'Unserved electricity (%)': 100 * simulation_results['UnservedElectricityMWh'].sum() / electricity_demand_mwh,
    'Hours with unserved electricity': int((simulation_results['UnservedElectricityMWh'] > 0).sum()),
    'Truck methanol demand (PJ)': mwh_to_pj(methanol_demand_mwh),
    'Unserved methanol (MWh)': simulation_results['UnservedMethanolMWh'].sum(),
    'Unserved methanol (%)': 100 * simulation_results['UnservedMethanolMWh'].sum() / methanol_demand_mwh,
    'Hours with unserved methanol': int((simulation_results['UnservedMethanolMWh'] > 0).sum()),
    'P2X electricity input (PJ)': mwh_to_pj(p2x_input_mwh),
    'P2X input capacity (MW)': SCENARIO['p2x_input_capacity_mw'],
    'P2X capacity factor (%)': (
        100 * p2x_input_mwh / (SCENARIO['p2x_input_capacity_mw'] * len(simulation_results))
        if SCENARIO['p2x_input_capacity_mw'] > 0 else 0.0
    ),
    'Methanol produced (PJ)': mwh_to_pj(methanol_production_mwh),
    'Biomass fuel used (PJ)': mwh_to_pj(biomass_fuel_used_mwh),
    'Biomass allowance remaining (PJ)': max(0.0, SCENARIO['biomass_available_pj'] - mwh_to_pj(biomass_fuel_used_mwh)),
    'Biomass allowance used (%)': (
        100 * mwh_to_pj(biomass_fuel_used_mwh) / SCENARIO['biomass_available_pj']
        if SCENARIO['biomass_available_pj'] > 0 else 0.0
    ),
    'Biomass plant capacity (MW)': SCENARIO['biomass_power_capacity_mw'],
    'Renewable curtailment (PJ)': mwh_to_pj(simulation_results['RenewableCurtailmentMWh'].sum()),
    'Battery power capacity (MW)': SCENARIO['battery_power_capacity_mw'],
    'Battery energy capacity (MWh)': SCENARIO['battery_energy_capacity_mwh'],
    'Battery charge input (PJ)': mwh_to_pj(simulation_results['BatteryChargeInputMWh'].sum()),
    'Battery discharge output (PJ)': mwh_to_pj(simulation_results['BatteryDischargeOutputMWh'].sum()),
    'Battery conversion losses (PJ)': mwh_to_pj(
        simulation_results['BatteryChargeInputMWh'].sum() * (1 - SCENARIO['battery_charge_efficiency'])
        + simulation_results['BatteryDischargeOutputMWh'].sum()
        * (1 / SCENARIO['battery_discharge_efficiency'] - 1)
    ),
    'Battery initial SOC (MWh)': simulation_results['BatterySOCStartMWh'].iloc[0],
    'Battery ending SOC (MWh)': simulation_results['BatterySOCEndMWh'].iloc[-1],
    'Battery net SOC change (MWh)': (
        simulation_results['BatterySOCEndMWh'].iloc[-1]
        - simulation_results['BatterySOCStartMWh'].iloc[0]
    ),
    'Methanol tank capacity (MWh)': SCENARIO['methanol_tank_capacity_mwh'],
    'Methanol initial inventory (MWh)': simulation_results['MethanolInventoryStartMWh'].iloc[0],
    'Methanol ending inventory (MWh)': simulation_results['MethanolInventoryEndMWh'].iloc[-1],
    'Methanol net inventory change (MWh)': (
        simulation_results['MethanolInventoryEndMWh'].iloc[-1]
        - simulation_results['MethanolInventoryStartMWh'].iloc[0]
    ),
})

display(summary.to_frame('value').round(3))

In [ ]:
plot_hours = min(24 * 14, len(simulation_results))
window = simulation_results.iloc[:plot_hours]
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(window['HourDK'], window['RenewableToLoadMWh'], label='Renewables to direct load')
axes[0].plot(window['HourDK'], window['ElectricityDemandMWh'], label='Electricity demand')
axes[0].plot(window['HourDK'], window['P2XElectricityInputMWh'], label='P2X electricity input')
axes[0].set_ylabel('MWh per hour')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(window['HourDK'], window['BatterySOCEndMWh'], color='tab:orange')
axes[1].set_ylabel('Battery SOC (MWh)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(window['HourDK'], window['MethanolDemandMWh'], label='Methanol demand')
axes[2].plot(window['HourDK'], window['MethanolProductionMWh'], label='Methanol production')
axes[2].set_ylabel('MWh fuel per hour')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

axes[3].plot(window['HourDK'], window['MethanolInventoryEndMWh'], color='tab:green')
axes[3].axhline(SCENARIO['methanol_tank_capacity_mwh'], color='tab:red', linestyle='--', label='Tank capacity')
axes[3].set_ylabel('Methanol inventory (MWh)')
axes[3].legend()
axes[3].grid(True, alpha=0.3)
axes[3].set_xlabel('Local time (HourDK)')

fig.suptitle('First two weeks of hourly dispatch')
fig.tight_layout()
plt.show()